In [6]:
# ============================================================
# MOVIE RECOMMENDATION DEMO
# Content-Based Filtering — MovieLens 1M
# ============================================================

import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 1. LOAD MOVIE DATA
# ------------------------------------------------------------

MOVIES_PATH = r"D:\ADA_Netflix\Neflix-Recommendation\movies.dat" # change path if required

movies = pd.read_csv(
    MOVIES_PATH,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["MovieID", "Title", "Genres"]
)

print(f"Loaded {len(movies):,} movies.")


# ------------------------------------------------------------
# 2. EXTRACT RELEASE YEAR
# ------------------------------------------------------------

movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)").astype(float)

# Remove year from title for cleaner display/search
movies["CleanTitle"] = (
    movies["Title"]
    .str.replace(r"\s*\(\d{4}\)", "", regex=True)
    .str.strip()
)


# ------------------------------------------------------------
# 3. CREATE CONTENT FEATURES
#    18 genre features + normalized release year
# ------------------------------------------------------------

movies["GenreList"] = movies["Genres"].apply(
    lambda x: x.split("|") if x != "(no genres listed)" else []
)

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies["GenreList"])

# Normalize release year
year_values = movies["Year"].fillna(movies["Year"].median()).values.reshape(-1, 1)

scaler = MinMaxScaler()
year_feature = scaler.fit_transform(year_values)

# Combine genres + year
content_features = np.hstack([
    genre_matrix,
    year_feature
])

print(f"Content feature matrix: {content_features.shape}")


# ------------------------------------------------------------
# 4. COSINE SIMILARITY
# ------------------------------------------------------------

similarity_matrix = cosine_similarity(content_features)

print("Content-based similarity matrix created.")


# ------------------------------------------------------------
# 5. MOVIE SEARCH
# ------------------------------------------------------------

def find_movie(movie_name):
    """
    Find the closest matching movie title.
    """

    query = movie_name.lower().strip()

    # Exact match
    exact = movies[
        movies["CleanTitle"].str.lower() == query
    ]

    if len(exact) > 0:
        return exact.iloc[0]

    # Partial match
    partial = movies[
        movies["CleanTitle"].str.lower().str.contains(
            re.escape(query),
            na=False
        )
    ]

    if len(partial) > 0:
        return partial.iloc[0]

    return None


# ------------------------------------------------------------
# 6. RECOMMENDATION FUNCTION
# ------------------------------------------------------------

def recommend_movies(movie_name, n=10):

    movie = find_movie(movie_name)

    if movie is None:
        print(f"\nMovie '{movie_name}' not found.")
        print("\nTry a title such as:")
        print(movies["CleanTitle"].sample(5, random_state=42).tolist())
        return

    movie_index = movie.name

    # Similarity scores for selected movie
    scores = similarity_matrix[movie_index]

    # Sort highest similarity first
    similar_indices = np.argsort(scores)[::-1]

    # Remove the input movie itself
    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ]

    # Top N
    top_indices = similar_indices[:n]

    recommendations = movies.iloc[top_indices].copy()
    recommendations["Similarity"] = scores[top_indices]

    print("\n" + "=" * 70)
    print(f"INPUT MOVIE: {movie['CleanTitle']} ({int(movie['Year'])})")
    print(f"GENRES: {movie['Genres']}")
    print("=" * 70)

    print("\nTOP 10 SIMILAR MOVIES:\n")

    for rank, (_, rec) in enumerate(
        recommendations.iterrows(), start=1
    ):
        print(
            f"{rank:2}. {rec['CleanTitle']} "
            f"({int(rec['Year'])})"
        )
        print(
            f"    Genres: {rec['Genres']} | "
            f"Similarity: {rec['Similarity']:.3f}"
        )

    return recommendations[
        ["MovieID", "CleanTitle", "Year", "Genres", "Similarity"]
    ]


# ------------------------------------------------------------
# 7. USER INPUT
# ------------------------------------------------------------

movie_name = input("\nEnter a movie name: ")

recommend_movies(movie_name)

Loaded 3,883 movies.
Content feature matrix: (3883, 19)
Content-based similarity matrix created.

INPUT MOVIE: Godfather, The (1972)
GENRES: Action|Crime|Drama

TOP 10 SIMILAR MOVIES:

 1. Godfather: Part II, The (1974)
    Genres: Action|Crime|Drama | Similarity: 1.000
 2. Untouchables, The (1987)
    Genres: Action|Crime|Drama | Similarity: 0.996
 3. Godfather: Part III, The (1990)
    Genres: Action|Crime|Drama | Similarity: 0.994
 4. Menace II Society (1993)
    Genres: Action|Crime|Drama | Similarity: 0.992
 5. Dead Presidents (1995)
    Genres: Action|Crime|Drama | Similarity: 0.991
 6. Limey, The (1999)
    Genres: Action|Crime|Drama | Similarity: 0.988
 7. Batman (1989)
    Genres: Action|Adventure|Crime|Drama | Similarity: 0.884
 8. Lethal Weapon 2 (1989)
    Genres: Action|Comedy|Crime|Drama | Similarity: 0.884
 9. Lethal Weapon 3 (1992)
    Genres: Action|Comedy|Crime|Drama | Similarity: 0.884
10. Man Bites Dog (C'est arrivé près de chez vous) (1992)
    Genres: Action|Comed

,MovieID,CleanTitle,Year,Genres,Similarity
1203,1221,"Godfather: Part II, The",1974.0,Action|Crime|Drama,0.999923
2125,2194,"Untouchables, The",1987.0,Action|Crime|Drama,0.995941
1954,2023,"Godfather: Part III, The",1990.0,Action|Crime|Drama,0.994249
489,493,Menace II Society,1993.0,Action|Crime|Drama,0.992301
41,42,Dead Presidents,1995.0,Action|Crime|Drama,0.990867
2843,2912,"Limey, The",1999.0,Action|Crime|Drama,0.987695
588,592,Batman,1989.0,Action|Adventure|Crime|Drama,0.883862
1932,2001,Lethal Weapon 2,1989.0,Action|Comedy|Crime|Drama,0.883862
1933,2002,Lethal Weapon 3,1992.0,Action|Comedy|Crime|Drama,0.883804
3197,3266,Man Bites Dog (C'est arrivé près de chez vous),1992.0,Action|Comedy|Crime|Drama,0.883804
